# Week 1 Day 6 — Custom `Dataset` and `DataLoader`
**Jul 6, 2026**

Day 5 used `TensorDataset` — a convenience wrapper around tensors you already had in memory. Today you build the actual `Dataset` contract yourself (just two methods), add a preprocessing transform, and handle the train/val split correctly — including a subtle bug that's easy to introduce by accident: leaking validation-set statistics into your normalization.

Scaffold as usual: TODO stubs, hints not formulas, self-check cells.

## Part 1: Data — synthetic tabular "credit risk" features

Given. Four continuous features (think: leverage, volatility, size, profitability) generating a binary default/no-default label through a noisy linear boundary — plain tabular data, not images, since that's the shape of data a custom `Dataset` most often wraps in practice.

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split

torch.manual_seed(0)
n_samples, n_features = 500, 4

X = torch.randn(n_samples, n_features)
true_w = torch.tensor([0.8, -1.2, 0.5, 1.0])
true_b = -0.2
logits = X @ true_w + true_b + 0.3 * torch.randn(n_samples)
y = (logits > 0).float()

print(X.shape, y.shape, "positive rate:", y.mean().item())

torch.Size([500, 4]) torch.Size([500]) positive rate: 0.4339999854564667


## Part 2: A custom `Dataset`

TODO: implement `TabularDataset(Dataset)`. The `Dataset` contract is exactly two methods:

- `__len__(self)` — how many samples total.
- `__getitem__(self, idx)` — return **one** sample, `(features, label)`, not a batch. Batching is `DataLoader`'s job, not yours.

Also accept an optional `transform` in `__init__` and store it. If it's set, apply it to the features (not the label) inside `__getitem__`, right before returning.

In [2]:
class TabularDataset(Dataset):
    def __init__(self, X, y, transform=None):
        # TODO: store X, y, transform
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        # TODO
        self.X.shape[0]
        return self.X.shape[0]

    def __getitem__(self, idx):
        # TODO: return (features, label) for a single sample, applying self.transform if set
        x = self.X[idx]
        y = self.y[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

# self-check
ds = TabularDataset(X, y)
assert len(ds) == n_samples, f"expected {n_samples}, got {len(ds)}"
xi, yi = ds[0]
assert xi.shape == (n_features,), f"expected ({n_features},), got {tuple(xi.shape)}"
assert yi.shape == (), f"expected scalar label, got {tuple(yi.shape)}"
print("Dataset OK:", len(ds), "samples")

Dataset OK: 500 samples


## Part 3: Train/val split — and a leakage trap

TODO: split `ds` into train/val subsets with `random_split` (pick a ratio, e.g. 80/20), using a seeded `torch.Generator` so the split is reproducible.

`random_split` returns `Subset` objects wrapping the *same* underlying `ds` — `train_subset.indices` gives you which rows landed in train. You'll need that in Part 4: the correct way to compute normalization statistics is **from the training rows only**. Computing mean/std over the full dataset (train + val combined) before splitting is a real, easy-to-miss bug — it leaks information about your validation set's distribution into training, similar in spirit to using future data in a point-in-time factor calculation.

In [3]:
# TODO: n_train, n_val = ...
n_train = int(0.8 * n_samples)
n_val = n_samples - n_train
# TODO: generator = torch.Generator().manual_seed(...)
generator = torch.Generator().manual_seed(42)
# TODO: train_subset, val_subset = random_split(ds, [...], generator=generator)
train_subset, val_subset = random_split(ds, [n_train, n_val], generator=generator)

print("train size:", len(train_subset), " val size:", len(val_subset))

train size: 400  val size: 100


## Part 4: A standardization transform, fit on train only

TODO: write a `Standardize` callable that z-score normalizes a single feature vector: `(x - mean) / std`. It should be constructed with a precomputed `mean` and `std` (don't recompute them per-sample inside `__call__`).

Then: compute `mean`/`std` from `ds.X` indexed by `train_subset.indices` only (**not** all of `ds.X`), build the `Standardize` transform from those, and attach it to `ds.transform`. Since both subsets wrap the same `ds` object, this one assignment makes both `train_subset` and `val_subset` apply the same train-fitted normalization — which is exactly what you want.

In [4]:
class Standardize:
    def __init__(self, mean, std):
        # TODO
        self.mean = mean
        self.std = std

    def __call__(self, x):
        # TODO
        return (x - self.mean) / self.std
        
train_X = ds.X[train_subset.indices]
# TODO: train_X = ds.X[...]
mean = train_X.mean(dim=0)
# TODO: mean = ...
std = train_X.std(dim=0)
# TODO: std = ...
# TODO: ds.transform = Standardize(mean, std)
ds.transform = Standardize(mean, std)

xi, yi = ds[train_subset.indices[0]]
print("normalized sample:", xi)

normalized sample: tensor([ 0.3503,  0.1093, -0.1911,  0.3710])


## Part 5: `DataLoader`s

TODO: build a `DataLoader` for `train_subset` (shuffled) and one for `val_subset` (not shuffled — there's never a reason to shuffle validation data, since order doesn't affect anything you compute from it). Pick a `batch_size`.

In [5]:
# TODO: train_loader = DataLoader(...)
# TODO: val_loader = DataLoader(...)
train_subset, val_subset = random_split(ds, [n_train, n_val], generator=generator)
train_loader = DataLoader(train_subset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=32, shuffle=False)
xb, yb = next(iter(train_loader))
print("train batch:", xb.shape, yb.shape)
xb, yb = next(iter(val_loader))
print("val batch:  ", xb.shape, yb.shape)

train batch: torch.Size([32, 4]) torch.Size([32])
val batch:   torch.Size([32, 4]) torch.Size([32])


## Part 6: Sanity-check the pipeline end to end

Given — a small pre-built model and training loop, not an exercise. The point here isn't the model (that was Day 5); it's confirming your `Dataset`/`DataLoader`/transform pipeline actually feeds a model correctly.

In [6]:
import torch.nn as nn

model = nn.Sequential(nn.Linear(n_features, 16), nn.ReLU(), nn.Linear(16, 1))
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(20):
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb).squeeze(-1), yb)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    correct, total = 0, 0
    for xb, yb in val_loader:
        preds = (model(xb).squeeze(-1) > 0).float()
        correct += (preds == yb).sum().item()
        total += yb.shape[0]

print(f"val accuracy: {correct / total:.3f}")

val accuracy: 0.930


## Try yourself

1. Deliberately reintroduce the leakage bug: compute `mean`/`std` from all of `ds.X` instead of just `train_subset.indices`, rerun Part 6. Does val accuracy change much on this dataset? (It may not, on data this clean — but note *why* the risk exists even when the visible damage is small, since real tabular data is rarely this clean.)
2. Add a second transform, e.g. clip extreme values before standardizing, and chain it with `Standardize` inside `__getitem__`.
3. Try `batch_size=1` vs. a large batch (e.g. 400) in `train_loader` — how does training stability/speed change?
4. Add a `sample_weight` per row (e.g. weight recent rows higher) returned as a third element from `__getitem__`, and use it to weight the loss per-sample in Part 6's loop instead of `criterion`'s default unweighted mean.